In [80]:
import pandas as pd

loans = pd.read_csv("../data/library_loans_clean.csv", parse_dates=["loan_date", "return_date"])
members_raw = pd.read_csv("../data/library_members.csv")
members = members_raw.copy()
members["birth_year"] = members["birth_year"].where(members["birth_year"].between(1920, 2015))
members = members.drop_duplicates()

# Task 1

In [81]:
has_member = loans["member_id"].isin(members["member_id"])
has_member.value_counts()

member_id
True     569
False     31
Name: count, dtype: int64

In [82]:
loans_with_missing_members = loans[~has_member]

loans_with_missing_members_count = loans_with_missing_members.shape[0]
all_loans_count = loans.shape[0]
loans_with_missing_members_share = loans_with_missing_members_count / all_loans_count

print("Loans with missing members count", loans_with_missing_members_count)
print("Loans with missing members, percentage share of all loans", 
      round(loans_with_missing_members_share, 3) * 100)


missing_members_sum = loans_with_missing_members["late_fee"].sum()
all_loans_sum = loans["late_fee"].sum()
loans_with_missing_members_share_late_fees = missing_members_sum / all_loans_sum

print("Late fee sum for loans with missing members", missing_members_sum)
print("Late fee sum for all loans", all_loans_sum)
print("Loans with missing members late fee, percentage share of all loans", 
      round(loans_with_missing_members_share_late_fees, 3) * 100)

Loans with missing members count 31
Loans with missing members, percentage share of all loans 5.2
Late fee sum for loans with missing members 347.5
Late fee sum for all loans 7487.5
Loans with missing members late fee, percentage share of all loans 4.6


In [83]:
genres_missing_members = loans_with_missing_members["genre"]

print("Value counts, genre for loans with missing members")
print(genres_missing_members.value_counts())

print("Value counts, genre for loans with missing members (shares)")
print(genres_missing_members.value_counts(normalize=True))

print("Value counts, all loans (shares)")
print(loans["genre"].value_counts(normalize=True))

Value counts, genre for loans with missing members
genre
Children    10
Sci-fi       5
Crime        4
Fantasy      4
Romance      4
History      4
Name: count, dtype: int64
Value counts, genre for loans with missing members (shares)
genre
Children    0.322581
Sci-fi      0.161290
Crime       0.129032
Fantasy     0.129032
Romance     0.129032
History     0.129032
Name: proportion, dtype: float64
Value counts, all loans (shares)
genre
Crime       0.258333
Children    0.216667
Sci-fi      0.148333
Fantasy     0.145000
Romance     0.136667
History     0.095000
Name: proportion, dtype: float64


The genre distribution of the loans with missing members compared to all loans differs slightly.  
But the number of missing loans is quite small (31) so it does not have enough datapoints to make a viable comparison.  

I would ask the library staff or the table maintainer of the member table if there are any known reasons of the missing members.

# Task 2

In [84]:
merged = loans.merge(members, on="member_id", how="outer", indicator=True)
merged["_merge"].value_counts()

_merge
both          569
left_only      31
right_only      1
Name: count, dtype: int64

In [85]:
members["member_id"].isin(loans["member_id"]).value_counts()

member_id
True     143
False      1
Name: count, dtype: int64

One member has never borrowed any books.  
Out of the two ways of finding the answer I prefer the second one with `isin()` as I find it more readable.

# Task 3

In [ ]:
merged = merged.query("_merge != 'right_only'")
merged["age_at_loan"] = (merged["loan_date"].dt.year - merged["birth_year"])

print(merged["age_at_loan"].describe())
print(merged.groupby("genre")["age_at_loan"].agg(["count", "mean"]))
print("Loans without age", merged["age_at_loan"].isna().sum())

count    457.000000
mean      38.770241
std       11.717328
min       16.000000
25%       30.000000
50%       39.000000
75%       47.000000
max       69.000000
Name: age_at_loan, dtype: float64
          count       mean
genre                     
Children    101  41.108911
Crime       120  38.408333
Fantasy      67  39.835821
History      45  35.222222
Romance      58  38.120690
Sci-fi       66  37.757576
Loans without age 143
